## TC 5033
## Deep Learning
## Transformers

#### Activity 4: Implementing a Translator

### Nombres: Freddy Silva, César Cruz, Fernando Guevara, Kailin Wu
### Matrículas: A00828792, A00825747, A00828723, A00830574

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



#### Script to convert csv to text file

In [5]:
#This script requires to convert the TSV file to CSV
# easiest way is to open it in Calc or excel and save as csv
PATH = './eng_to_spa.tsv'
import pandas as pd
df = pd.read_csv(PATH, sep="\t", on_bad_lines="skip", header=None, names=["id_en", "sentence_en", "id_es", "sentence_es"])

In [7]:
eng_spa_cols = df.iloc[:, [1, 3]]
eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()
eng_spa_cols = eng_spa_cols.sort_values(by='length')
eng_spa_cols = eng_spa_cols.drop(columns=['length'])

output_file_path = './eng_to_spa.txt'
eng_spa_cols.to_csv(output_file_path, sep='\t', index=False, header=False)

/tmp/ipykernel_5012/2215269847.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()


## Transformer - Attention is all you need

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [10]:
MAX_SEQ_LEN = 128

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Implements the sinusoidal positional encoding to inject relative or absolute 
    positional information into the sequence.

    Attributes:
        pos_embed_matrix (torch.Tensor): Fixed weight matrix containing sine and 
            cosine functions of different frequencies.
    """
    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0)/d_model))
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input embeddings of shape (batch_size, seq_len, d_model).
        
        Returns:
            torch.Tensor: Embeddings with positional information added.
        """
#         print(self.pos_embed_matrix.shape)
#         print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    """
    Computes Multi-Head Attention as described in 'Attention Is All You Need'.
    
    Args:
        d_model (int): The dimension of the model (embedding size).
        num_heads (int): Number of parallel attention heads.
    """
    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'

        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask = None):
        """
        Args:
            Q, K, V (torch.Tensor): Input tensors of shape (batch_size, seq_len, d_model).
            mask (torch.Tensor, optional): Mask to prevent attention to certain positions.
        
        Returns:
            tuple: (output_tensor, attention_weights)
        """
        batch_size = Q.size(0)
        '''
        Q, K, V -> [batch_size, seq_len, num_heads*d_k]
        after transpose Q, K, V -> [batch_size, num_heads, seq_len, d_k]
        '''
        # Linear projection and split into heads
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )

        
        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        # Concatenate heads and apply final linear layer
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)

        return weighted_values, attention


    def scale_dot_product(self, Q, K, V, mask = None):
        """Calculates Scaled Dot-Product Attention."""
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)

        return weighted_values, attention


class PositionFeedForward(nn.Module):
    """
    Implements a position-wise fully connected feed-forward network.
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderSubLayer(nn.Module):
    """
    A single layer of the Encoder consisting of self-attention and FFN.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    """
    Transformer Encoder consisting of a stack of N EncoderSubLayers.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):
    """
    A single layer of the Decoder including Self-Attention, Cross-Attention, and FFN.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)

        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)

        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)

class Decoder(nn.Module):
    """
    Transformer Decoder consisting of a stack of N DecoderSubLayers.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, target_mask, encoder_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

In [ ]:
class Transformer(nn.Module):
    This is the final "boss" class that orchestrates everything. I've documented the full flow, including the specific logic for the Causal Mask (the triangular mask) and the Padding Mask, which are vital for a Transformer to work correctly during training.

I also added a small fix to the mask method to ensure it handles the device correctly, similar to the improvements we made in the previous steps.

Documented Transformer Class
Python
class Transformer(nn.Module):
    """
    Complete Transformer model for sequence-to-sequence tasks.
    
    This class integrates the Encoder, Decoder, and Positional Embeddings 
    to process input sequences and generate target sequences.

    Args:
        d_model (int): Dimension of the embedding vectors.
        num_heads (int): Number of attention heads.
        d_ff (int): Hidden layer dimension of the feed-forward network.
        num_layers (int): Number of encoder and decoder sub-layers to stack.
        input_vocab_size (int): Size of the source vocabulary.
        target_vocab_size (int): Size of the target vocabulary.
        max_len (int): Maximum sequence length for positional encoding.
        dropout (float): Dropout probability
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size,
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)

    def forward(self, source, target):
        """
        Processes source and target sequences through the Transformer.

        Args:
            source (torch.Tensor): Source indices of shape (batch_size, src_seq_len).
            target (torch.Tensor): Target indices of shape (batch_size, tgt_seq_len).

        Returns:
            torch.Tensor: Logits for the target vocabulary of shape 
                (batch_size, tgt_seq_len, target_vocab_size).
        """
        # Encoder mask
        source_mask, target_mask = self.mask(source, target)
        # Embedding and positional Encoding
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)

        # Decoder embedding and postional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        # Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)

        return self.output_layer(output)



    def mask(self, source, target):
        """
        Creates masks to prevent the model from attending to padding tokens 
        and future tokens in the target sequence.

        Args:
            source (torch.Tensor): Source sequence tensor.
            target (torch.Tensor): Target sequence tensor.

        Returns:
            tuple: (source_mask, target_mask) where target_mask includes 
                the look-ahead (triangular) constraint.
        """
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask
        return source_mask, target_mask


#### Simple test

In [13]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [14]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size,
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [15]:
output = model(source, target)

In [16]:
# Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


### Translator Eng-Spa

In [17]:
PATH = './eng_to_spa.txt'

In [18]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

In [19]:
eng_spa_pairs[:10]

[['Go!', '¡Vete!'],
 ['No.', 'No.'],
 ['No!', '¡No!'],
 ['Hi.', 'Hola.'],
 ['So?', '¿Y?'],
 ['Go.', 'Vayan.'],
 ['Go.', 'Váyanse.'],
 ['Me?', '¿Yo?'],
 ['OK.', 'Bueno.'],
 ['Go.', 'Váyase.']]

In [20]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [21]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go!', 'No.', 'No!', 'Hi.', 'So?', 'Go.', 'Go.', 'Me?', 'OK.', 'Go.']
['¡Vete!', 'No.', '¡No!', 'Hola.', '¿Y?', 'Vayan.', 'Váyanse.', '¿Yo?', 'Bueno.', 'Váyase.']


In [ ]:
def preprocess_sentence(sentence):
    """
    Cleans and formats a raw sentence for NLP tasks by removing special 
    characters, normalizing accents, and adding sequence tokens.

    Args:
        sentence (str): The raw input string to be processed.

    Returns:
        str: The cleaned sentence, lowercased, stripped of non-alphabetic 
            characters, with normalized vowels and wrapped in <sos>/<eos> tokens.
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [23]:
s1 = '¿Hola @ cómo estás? 123'

In [24]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [25]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [26]:
spa_sentences[:10]

['<sos> vete <eos>',
 '<sos> no <eos>',
 '<sos> no <eos>',
 '<sos> hola <eos>',
 '<sos> y <eos>',
 '<sos> vayan <eos>',
 '<sos> vayanse <eos>',
 '<sos> yo <eos>',
 '<sos> bueno <eos>',
 '<sos> vayase <eos>']

In [ ]:
def build_vocab(sentences):
    """
    Constructs a bidirectional mapping between words and unique integer indices 
    based on word frequency.

    The function reserves index 0 for padding and index 1 for unknown tokens, 
    then assigns indices to words starting from index 2, ordered by frequency.

    Args:
        sentences (list of str): A list of preprocessed strings.

    Returns:
        tuple: A pair (word2idx, idx2word) containing:
            - word2idx (dict): Maps string tokens to integer indices.
            - idx2word (dict): Maps integer indices back to string tokens.
    """
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [28]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [29]:
print(eng_vocab_size, spa_vocab_size)

28410 48277


In [ ]:
class EngSpaDataset(Dataset):
    """
    A custom PyTorch Dataset for English-to-Spanish translation.

    This class handles the conversion of preprocessed strings into integer tensors 
    on-the-fly, allowing for efficient batching during model training.

    Attributes:
        eng_sentences (list of str): List of preprocessed English source sentences.
        spa_sentences (list of str): List of preprocessed Spanish target sentences.
        eng_word2idx (dict): Vocabulary mapping for the English language.
        spa_word2idx (dict): Vocabulary mapping for the Spanish language.
    """
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]

        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

In [ ]:
def collate_fn(batch):
    """
    Custom collation function to handle batches of variable-length sequences.
    This function truncates sequences to a maximum length and pads shorter 
    sequences with zeros so they can be stacked into a single tensor.

    Args:
        batch (list of tuple): A list of (source_tensor, target_tensor) pairs 
            provided by the Dataset's __getitem__ method.

    Returns:
        tuple: (eng_batch, spa_batch) where both are padded tensors of shape 
            (batch_size, max_seq_len_in_batch).
    """
    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch]
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch


In [ ]:
def train(model, dataloader, loss_function, optimiser, epochs):
    """
    Executes the training loop for the Transformer model using Teacher Forcing.

    Args:
        model (nn.Module): The Transformer model to be trained.
        dataloader (DataLoader): PyTorch DataLoader providing batched (src, tgt) tensors.
        loss_function (callable): The criterion (e.g., CrossEntropyLoss) to calculate error.
        optimiser (torch.optim.Optimizer): The optimizer (e.g., Adam) to update weights.
        epochs (int): Number of complete passes over the dataset.
    """
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Decoder preprocessing
            target_input = spa_batch[:, :-1]
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output)
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()

        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')



In [33]:
BATCH_SIZE = 64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [34]:
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [35]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [36]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

Epoch: 0/10, Loss: 3.5568
Epoch: 1/10, Loss: 2.1690
Epoch: 2/10, Loss: 1.6772
Epoch: 3/10, Loss: 1.3573
Epoch: 4/10, Loss: 1.1121
Epoch: 5/10, Loss: 0.9147
Epoch: 6/10, Loss: 0.7549
Epoch: 7/10, Loss: 0.6285
Epoch: 8/10, Loss: 0.5361
Epoch: 9/10, Loss: 0.4707


In [ ]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    """
    Translates a single English sentence into Spanish using the trained Transformer.

    This function implements Greedy Decoding, where the model predicts the next 
    token based on the source sentence and the tokens generated so far.

    Args:
        model (nn.Module): The trained Transformer model.
        sentence (str): The raw English sentence to translate.
        eng_word2idx (dict): English vocabulary mapping.
        spa_idx2word (dict): Spanish inverse vocabulary mapping.
        max_len (int): Maximum number of tokens to generate.
        device (str/torch.device): The device to run inference on.

    Returns:
        str: The translated Spanish sentence.
    """
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

In [40]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!",
    "I went to the store because I needed to buy some food.",
    "She likes coffee, but she prefers tea in the morning.",
    "I have finished my homework.",
    "If I had more time, I would learn another language.",
    "Maria told Juan that she would call him later.",
    "By the way, have you seen my phone?"
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)


Input sentence: Hello, how are you?
Traducción: <sos> hola que tal <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo inteligencia artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia artificial es buena <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: I went to the store because I needed to buy some food.
Traducción: <sos> fui a la tienda porque necesitaba algo de comida <eos>

Input sentence: She likes coffee, but she prefers tea in the morning.
Traducción: <sos> a ella le gusta el te pero el cafe a la ma ana <eos>

Input sentence: I have finished my homework.
Traducción: <sos> termine los deberes <eos>

Input sentence: If I had more time, I would learn another language.
Traducción: <sos> si tuviera mas tiempo aprenderia otro idioma <eos>

Input sentence: Maria told Juan that she would call him later.
Traducción: <sos> maria le dijo que llamaria mas